# RolloTree Quickstart

This notebook demonstrates the basic usage of **RolloTree** — a rolling lookahead optimal classification tree.

We'll cover:
1. Loading data & training a model
2. Making predictions & evaluating accuracy
3. Class probability estimates (`predict_proba`)
4. Feature importances
5. Inspecting the fitted tree

## 1. Setup

```bash
pip install rollotree[fast]  # includes numba acceleration
```

In [1]:
import pandas as pd
import numpy as np
from rollotree import RollingOCT

## 2. Load Data

We'll use the bundled Wine dataset (binarized version of the [UCI Wine Dataset](https://archive.ics.uci.edu/dataset/109/wine)).

- **3 classes** (wine varieties)
- **130 binary features** (one-hot encoded from the original 13 continuous features)
- Target column: `y`

In [2]:
train = pd.read_csv("../rollotree/data/train.csv")
test = pd.read_csv("../rollotree/data/test.csv")

print(f"Training samples: {len(train)}")
print(f"Test samples:     {len(test)}")
print(f"Features:         {train.shape[1] - 1}")
print(f"Classes:          {sorted(train['y'].unique())}")
train.head()

Training samples: 160
Test samples:     18
Features:         130
Classes:          [1, 2, 3]


,y,1,2,3,4,5,6,7,8,9,...,121,122,123,124,125,126,127,128,129,130
0,1,0,0,0,0,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [3]:
X_train = train.drop("y", axis=1)
y_train = train["y"]

X_test = test.drop("y", axis=1)
y_test = test["y"]

print(f"Class distribution (train): {y_train.value_counts().to_dict()}")
print(f"Class distribution (test):  {y_test.value_counts().to_dict()}")

Class distribution (train): {2: 64, 1: 53, 3: 43}
Class distribution (test):  {2: 7, 1: 6, 3: 5}


## 3. Train a Model

Create a `RollingOCT` classifier and call `fit()`. The API follows the familiar sklearn pattern.

Key parameters:
- `depth`: Maximum tree depth (>= 2)
- `criterion`: `"gini"` or `"misclassification"`
- `solver`: `"highs"` (default, open-source), `"gurobi"`, or `"cbc"`

In [4]:
# Train a depth-3 tree (one round of rolling expansion)
model = RollingOCT(depth=3, criterion="gini", solver="highs")
model.fit(X_train, y_train)

print(f"Train accuracy: {model.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {model.score(X_test, y_test):.3f}")

Train accuracy: 0.694
Test accuracy:  0.778


## 4. Predictions

In [5]:
predictions = model.predict(X_test)

print(f"Predictions shape: {predictions.shape}")
print(f"First 10 predictions: {predictions[:10]}")
print(f"First 10 actual:      {y_test.values[:10]}")

Predictions shape: (18,)
First 10 predictions: [1 1 2 2 2 1 2 2 2 2]
First 10 actual:      [1 1 1 1 1 1 2 2 2 2]


## 5. Class Probability Estimates

`predict_proba()` returns per-class probabilities based on the class distribution at each leaf node.

In [6]:
proba = model.predict_proba(X_test)

print(f"Shape: {proba.shape}  (n_samples x n_classes)")
print(f"Classes: {model.classes_}")
print()
print("First 5 samples:")
proba_df = pd.DataFrame(proba, columns=[f"class_{c}" for c in model.classes_])
proba_df.head()

Shape: (18, 3)  (n_samples x n_classes)
Classes: [1, 2, 3]

First 5 samples:


,class_1,class_2,class_3
0,1.000000,0.000000,0.00000
1,1.000000,0.000000,0.00000
2,0.342857,0.533333,0.12381
3,0.342857,0.533333,0.12381
4,0.342857,0.533333,0.12381


In [7]:
# Probabilities sum to 1.0 for each sample
print(f"Row sums (first 5): {proba[:5].sum(axis=1)}")
print(f"All rows sum to 1: {np.allclose(proba.sum(axis=1), 1.0)}")

Row sums (first 5): [1. 1. 1. 1. 1.]
All rows sum to 1: True


## 6. Feature Importances

`feature_importances_` shows how often each feature is used as a split, normalized to sum to 1.

In [8]:
importances = model.feature_importances_

print(f"Shape: {importances.shape}")
print(f"Sum:   {importances.sum():.6f}")

# Top 10 most important features
feature_names = X_train.columns
top_idx = np.argsort(importances)[::-1][:10]

print("\nTop 10 features:")
for i in top_idx:
    print(f"  {feature_names[i]:20s}  importance={importances[i]:.4f}")

Shape: (130,)
Sum:   1.000000

Top 10 features:
  31                    importance=0.2000
  112                   importance=0.2000
  50                    importance=0.2000
  21                    importance=0.2000
  111                   importance=0.2000
  38                    importance=0.0000
  39                    importance=0.0000
  40                    importance=0.0000
  37                    importance=0.0000
  42                    importance=0.0000


## 7. Per-depth Results

In [9]:
for depth, result in sorted(model.depth_results_.items()):
    print(
        f"Depth {depth}: "
        f"train_acc={result.training_accuracy:.3f}, "
        f"test_acc={result.test_accuracy:.3f}, "
        f"time={result.elapsed_time:.2f}s"
    )

Depth 2: train_acc=0.588, test_acc=0.588, time=1.67s
Depth 3: train_acc=0.694, test_acc=0.694, time=1.48s


## 8. Numpy Arrays Work Too

In [10]:
model_np = RollingOCT(depth=2, solver="highs")
model_np.fit(X_train.values, y_train.values)
print(f"Accuracy (numpy input): {model_np.score(X_test.values, y_test.values):.3f}")

Accuracy (numpy input): 0.611


## Summary

| Step | Code |
|------|------|
| Create model | `model = RollingOCT(depth=3, solver="highs")` |
| Train | `model.fit(X_train, y_train)` |
| Predict | `preds = model.predict(X_test)` |
| Probabilities | `proba = model.predict_proba(X_test)` |
| Importances | `model.feature_importances_` |
| Evaluate | `acc = model.score(X_test, y_test)` |
| Inspect tree | `model.tree_.branch_nodes`, `model.tree_.leaf_nodes` |
| Per-depth stats | `model.depth_results_` |

For tree visualization and inspection, see **02_visualization.ipynb**.  
For sklearn integration and model persistence, see **03_sklearn_integration.ipynb**.